# UrbanHub - Batch météo NOAA (Rémi)
Source : NOAA ISD Global Hourly, [archive publique NODD](https://registry.opendata.aws/noaa-global-hourly/). L’échantillon couvre Orly, Lyon–Saint-Exupéry et Lille–Lesquin en 2025, jusqu’au 24 août. Pour la France entière, le mode `france` traite par défaut les cinq dernières années civiles complètes d’ISD (2020 à 2024). La NOAA recommande [GHCNh](https://www.ncei.noaa.gov/products/global-historical-climatology-network-hourly) pour les observations récentes.

Le notebook ouvre par défaut les Parquets France déjà produits. Activez `REBUILD` pour reconstruire le lot localement. Pour revenir au test, sélectionnez `sample` et l’année 2025.

In [ ]:
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/noaa_batch.py').is_file())
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import pandas as pd
from scripts.noaa_batch import run

MODE = 'france'
START_YEAR, END_YEAR = 2020, 2024
REBUILD = False
output = root / 'data' / f'noaa-{MODE}-{START_YEAR}-{END_YEAR}'
if REBUILD or not (output / 'gold/kpis/weather_daily.parquet').is_file():
    output = run(MODE, START_YEAR, END_YEAR, workers=3 if MODE == 'sample' else 8)

## Couverture et qualité
Silver déduplique par station et instant UTC. Unités : °C, m/s, degrés, hPa, mètres et mm. Gold agrège d’abord chaque station par heure, puis par ville. Les jours sont en UTC. `precipitation_observed` est la somme des heures renseignées (`precipitation_hours`), donc un cumul partiel si ce nombre est inférieur à 24 ; une journée sans mesure reste nulle au sens Parquet (absence de valeur).

In [ ]:
manifest = pd.read_json(output / 'bronze/noaa/manifest.json', dtype={'station_id': str})
display(manifest.groupby(['year', 'status']).size().unstack(fill_value=0))
hourly = pd.read_parquet(output / 'gold/kpis/weather_hourly.parquet')
daily = pd.read_parquet(output / 'gold/kpis/weather_daily.parquet')
assert not hourly.duplicated(['city_id', 'hour_utc']).any()
display(hourly.groupby('city_id').agg(
    debut=('hour_utc', 'min'), fin=('hour_utc', 'max'), heures=('hour_utc', 'size'),
    temperature_disponible=('temperature', 'count'), pluie_disponible=('precipitation', 'count')))

## Évolution saisonnière et couverture nationale
En mode France, le profil national donne le même poids à chaque station du catalogue `FR` ayant des températures disponibles au mois considéré ; le nombre de stations est affiché. En mode échantillon, seules les trois villes sont représentées. La fin de série en 2025 est tronquée : l’absence des mois suivants n’indique pas un changement de climat.

In [ ]:
city_month = hourly.assign(month=hourly['hour_utc'].dt.strftime('%Y-%m'))
city_month.pivot_table(index='month', columns='city_id', values='temperature').plot(
    title='Température moyenne mensuelle - villes', ylabel='°C', xlabel='Mois UTC')
plt.show()
if MODE == 'france':
    station_months = []
    for path in sorted((output / 'silver/weather').glob('year=*/weather_france.parquet')):
        observations = pd.read_parquet(path, columns=['station_id', 'timestamp_utc', 'temperature'])
        observations['hour'] = observations['timestamp_utc'].dt.floor('h')
        station_hours = observations.groupby(['station_id', 'hour'], as_index=False)['temperature'].mean()
        station_hours['month'] = station_hours['hour'].dt.strftime('%Y-%m')
        station_months.append(station_hours.groupby(['station_id', 'month'], as_index=False)['temperature'].mean())
    national = pd.concat(station_months).groupby('month')['temperature'].agg(temperature='mean', stations='count')
    national.plot(subplots=True, title=['Température moyenne (°C)', 'Stations avec température'], sharex=True)
    plt.show()

## Extrêmes et jours inhabituels
On retient les jours avec au moins 18 heures de température. Par ville et saison, les maxima au-dessus du P95 et les minima sous le P5 signalent des extrêmes relatifs à la période observée. On ne calcule pas de seuil avec moins de 30 jours de référence. Ces seuils descriptifs ne constituent pas une normale climatique. Les jours inhabituels consécutifs peuvent signaler un épisode à examiner.

In [ ]:
valid = daily.loc[daily['temperature_hours'].ge(18)].copy()
valid['season'] = ((valid['day_utc'].dt.month % 12) // 3).map({0: 'DJF', 1: 'MAM', 2: 'JJA', 3: 'SON'})
groups = valid.groupby(['city_id', 'season'])
valid['reference_days'] = groups['day_utc'].transform('size')
valid['p95'] = groups['temperature_max'].transform('quantile', q=.95).where(valid['reference_days'].ge(30))
valid['p05'] = groups['temperature_min'].transform('quantile', q=.05).where(valid['reference_days'].ge(30))
hot = valid.loc[valid['temperature_max'].gt(valid['p95'])].assign(extreme='chaud')
cold = valid.loc[valid['temperature_min'].lt(valid['p05'])].assign(extreme='froid')
extremes = pd.concat([hot, cold]).sort_values(['city_id', 'extreme', 'day_utc'])
display(extremes[['city_id', 'day_utc', 'extreme', 'temperature_min', 'temperature_max', 'p05', 'p95', 'reference_days']].head(15))
gap = extremes.groupby(['city_id', 'extreme'])['day_utc'].diff()
episodes = (extremes.assign(episode=gap.ne(pd.Timedelta(days=1)).cumsum())
            .groupby(['city_id', 'extreme', 'episode'], as_index=False)
            .agg(debut=('day_utc', 'min'), fin=('day_utc', 'max'), jours=('day_utc', 'size')))
display(episodes.loc[episodes['jours'].ge(2)].drop(columns='episode').sort_values('jours', ascending=False).head(10))

## Météo et visibilité
Corrélation de Pearson par ville sur les paires effectivement mesurées (au moins 30 heures), avec l’effectif `n`. La pluie est bien moins renseignée que les autres variables ; ne pas remplacer les valeurs absentes par zéro. Les corrélations sont descriptives et ne prouvent pas un effet causal.

In [ ]:
correlations = []
for city, frame in hourly.groupby('city_id'):
    for variable in ['temperature', 'wind_speed', 'precipitation']:
        pairs = frame[[variable, 'visibility']].dropna()
        r = pairs[variable].corr(pairs['visibility']) if len(pairs) >= 30 and pairs.nunique().min() > 1 else None
        correlations.append({'city_id': city, 'variable': variable, 'n': len(pairs), 'pearson_r': r})
    plt.scatter(frame['temperature'], frame['visibility'], s=4, alpha=.2, label=city)
plt.xlabel('Température (°C)')
plt.ylabel('Visibilité (m)')
plt.legend()
plt.show()
display(pd.DataFrame(correlations).round(3))

## Bilan du lot France (2020–2024)
À l’échelle du réseau français, la moyenne mensuelle passe de 5,7 °C en janvier à 21,3 °C en août. Début septembre 2023, le seuil de chaleur saisonnier est dépassé 10 jours consécutifs à Lyon, 9 à Paris et 8 à Lille. La température est positivement associée à la visibilité dans les trois villes. Les valeurs de pluie retenues sont constantes (zéro) : aucune corrélation pluie–visibilité n’est interprétable. Voir le [bilan NOAA archivé](../docs/ARCHIVE_RESULTATS_NOAA.md) pour les chiffres, les limites et les chemins R2.